# Benchmark — Buril contra los mejores limpiadores de trazos (Google Colab)

Cuaderno **autónomo**: la lógica del benchmark va **embebida** aquí abajo, así
que **no clona el repo** ni importa `tools/benchmark_limpiar_trazos.py`. No
depende de qué rama tenga los archivos ni falla por importaciones: abrís, corrés
las celdas en orden y listo.

**Qué hace**: baja **Buril** de la release `modelo-buril` y el dataset `.npz` de
`dataset-buril` (ambos **anónimos**, sin token), toma N dibujos del
`split=validacion` y les pasa **los mismos** a cada modelo. Después normaliza la
salida de todos (**umbral + esqueleto Zhang-Suen**, para no comparar un esqueleto
de 1 px contra una línea gruesa) y mide IoU/Dice esqueleto↔esqueleto, IoU de
máscara, precisión/recall, **clDice** (topología) y **PSNR**, más latencia y peso.

**Rivales (5)** con los que se compara a Buril:

| Modelo | Tarea | Licencia |
|---|---|---|
| Simo-Serra et al. (`bobbens/sketch_simplification`, GAN) | limpieza de bocetos (referencia) | **CC-BY-NC-SA (no comercial)** |
| Anime2Sketch | extracción de línea | MIT (código) |
| Informative Drawings | generación de líneas (no apareada) | MIT (código) |
| ControlNet Lineart | lineart desde boceto | Apache-2.0 |
| PidiNet soft-edge | bordes suaves | Apache-2.0 |

Más dos referencias: **Zhang-Suen** puro (algoritmo) y, si la release tiene el
`.tflite`, **Buril**.

> **Aviso honesto**: Buril es un *especialista en esqueleto de 1 px* (parches
> 64×64, ~cientos de KB, corre en el móvil); los rivales son limpieza/extracción
> general en *full-image* con pesos de cientos de MB. El protocolo (mismo set +
> misma normalización) hace la comparación justa en *forma*, pero **no hay un
> ganador universal**: mirá todas las métricas, no sólo una. El modelo de
> Simo-Serra es **no comercial**; sirve como benchmark de investigación, no como
> argumento de venta.

**Antes de empezar**: `Entorno de ejecución → Cambiar tipo de entorno → GPU`
(opcional pero recomendado). No hace falta token ni clonar nada.

In [ ]:
# @title 1. Configuración (autónomo: no clona el repo)
REPO = "LuisAlejandro544/Dibujo-"           # @param {type:"string"}
TAG_MODELO = "modelo-buril"                 # @param {type:"string"}
TAG_DATOS = "dataset-buril"                 # @param {type:"string"}
VERSION_DATOS = "v1.0.0-experimental.1a"    # @param {type:"string"}
LADO = 512                                  # @param {type:"integer"}
POR_CATEGORIA = 100                         # @param {type:"integer"}
MUESTRAS = 8                                # @param {type:"integer"}
SEMILLA = 1                                 # @param {type:"integer"}
MODELOS = "buril,simoserra,anime2sketch,informative,lineart,pidinet,zhangsuen"  # @param {type:"string"}

import os

BURIL = ".cache/modelos/limpiar_trazos.tflite"
DIR = ".cache/limpiar_trazos_planb/procesado"
NPZ = f"{DIR}/pares_esq_lado{LADO}.npz"
print("Configuración lista. La lógica va embebida en la celda 4; no se clona nada.")


In [ ]:
# @title 2. Dependencias
import importlib, subprocess, sys

def _instalar(*paquetes):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)

for modulo, paquete in [
    ("numpy", "numpy"), ("PIL", "pillow"), ("skimage", "scikit-image"),
    ("matplotlib", "matplotlib"), ("requests", "requests"),
    ("tensorflow", "tensorflow"),
    ("torch", "torch"), ("torchvision", "torchvision"),
    ("controlnet_aux", "controlnet-aux"), ("huggingface_hub", "huggingface_hub"),
]:
    try:
        importlib.import_module(modulo)
    except ImportError:
        print("instalando", paquete, "…")
        _instalar(paquete)

import tensorflow as tf
import torch
print("TensorFlow", tf.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.is_available())


In [ ]:
# @title 3. Bajar Buril y el dataset desde las releases (anónimo, sin token)
import os
import requests

API = f"https://api.github.com/repos/{REPO}"


def assets(tag):
    r = requests.get(f"{API}/releases/tags/{tag}")
    return r.json().get("assets", []) if r.status_code == 200 else []


def bajar(tag, nombre, destino):
    url = f"https://github.com/{REPO}/releases/download/{tag}/{nombre}"
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        return False
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    print(f"Bajado {nombre} ({os.path.getsize(destino)} bytes) de {tag}")
    return True


if not bajar(TAG_MODELO, "limpiar_trazos.tflite", BURIL):
    print("AVISO: la release no tiene el modelo; Buril se saltea.")
    BURIL = None

ASSET = f"pares_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.npz"
disponibles = [a["name"] for a in assets(TAG_DATOS)]
objetivo = ASSET if ASSET in disponibles else next(
    (n for n in disponibles
     if n.startswith(f"pares_esq_lado{LADO}_pc") and n.endswith(".npz")), None)
if objetivo is None or not bajar(TAG_DATOS, objetivo, NPZ):
    raise SystemExit(
        f"No hay .npz en la release {TAG_DATOS}. "
        "Corré tools/buril_colab.ipynb para generarlo.")
print("Dataset:", NPZ, "|", os.path.getsize(NPZ), "bytes")


## Lógica embebida del benchmark

La siguiente celda define las funciones (`ejecutar`, `imprimir_ranking`, `dibujar_rejilla`, `autoprueba`, …). No la edites salvo que sepas lo que hacés: es la copia autónoma de `tools/benchmark_limpiar_trazos.py`.

In [ ]:
# @title 4. Lógica del benchmark (embebida, autónoma)
# -------------------------------------------------------------------------
# Lógica del benchmark, embebida para que este cuaderno sea autónomo:
# no clona el repo ni importa tools/benchmark_limpiar_trazos.py, así no
# depende de qué rama tenga el archivo. Copia de ese script (que sigue
# siendo la fuente para el CLI y `autoprueba`).
# -------------------------------------------------------------------------
import gc
import json
import os
import subprocess
import time

import numpy as np

RAIZ = os.getcwd()  # en Colab, /content; sólo para la caché de rivales

# Debe coincidir con `LimpiadorTrazosIa` (parche 64x64, paso 48 = solape 16).
LADO_PARQUE = 64
PASO_PARQUE = 48

# Umbral de "tinta" del repase de entrada (el fondo de la hoja es blanco puro).
UMBRAL_TINTA_ENTRADA = 250

LADO_DPI = 512

# Repos externos que se clonan una sola vez en `.cache/benchmark/`.
URL_ANIME2SKETCH = "https://github.com/Mukosame/Anime2Sketch"
PESOS_ANIME2SKETCH = "https://huggingface.co/public-data/Anime2Sketch/resolve/main/netG.pth"
REPO_SIMO_SERRA = "Rayonapp/sketch_simplification"  # pesos .t7 ya convertidos
ARCHIVO_SIMO_SERRA = "model_gan.pth"
REPO_INFORMATIVE = "carolineec/informativedrawings"  # Space con model.pth
ARCHIVO_INFORMATIVE = "model.pth"


# --------------------------------------------------------------------------
# Utilidades de imagen y métricas (numpy + scikit-image; sin modelos)
# --------------------------------------------------------------------------


def a_tinta(gray):
    """Devuelve la "tinta" (0..1) de una imagen gris, sin importar polaridad.

    La línea es lo que se aparta del fondo (la moda). Así funciona igual si el
    modelo pinta línea oscura sobre blanco o línea clara sobre negro.
    """
    g = np.asarray(gray, np.float32)
    if g.max() > 1.0:
        g = g / 255.0
    fondo = float(np.median(g))
    t = np.abs(g - fondo)
    mx = float(t.max())
    return (t / mx) if mx > 1e-6 else t


def binarizar(tinta, umbral=None):
    """Binariza una imagen de tinta; por defecto con Otsu (con respaldo fijo)."""
    from skimage.filters import threshold_otsu

    t = np.asarray(tinta, np.float32)
    if umbral is None:
        try:
            umbral = float(threshold_otsu(t))
        except Exception:
            umbral = 0.5
    return t > max(umbral, 1e-3)


def esqueletizar(mask):
    """Esqueleto de 1 px (Zhang-Suen). Idempotente sobre algo ya de 1 px."""
    from skimage.morphology import skeletonize

    return skeletonize(np.asarray(mask, bool))


def _iou(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = float(np.logical_or(a, b).sum())
    return float(np.logical_and(a, b).sum()) / max(union, 1.0)


def _dice(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    s = float(a.sum()) + float(b.sum())
    return 2.0 * float(np.logical_and(a, b).sum()) / max(s, 1.0)


def _precision_recall(pred, gt):
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    tp = float(np.logical_and(pred, gt).sum())
    return tp / max(float(pred.sum()), 1.0), tp / max(float(gt.sum()), 1.0)


def psnr(gt, pred):
    """PSNR entre la máscara GT (0/1) y la predicción normalizada."""
    gt = np.asarray(gt, np.float64)
    pred = np.asarray(pred, np.float64)
    mse = float(np.mean((gt - pred) ** 2))
    if mse <= 1e-12:
        return 99.0
    return float(10.0 * np.log10(1.0 / mse))


def cldice(gt, pred):
    """clDice (Shit et al., CVPR'21): preserva la topología del esqueleto.

    Con un GT de 1 px el IoU es durísimo y engaña; clDice premia que la
    predicción siga la conectividad correcta aunque el grosor no calce.
    """
    gt = np.asarray(gt, bool)
    pred = np.asarray(pred, bool)
    s_gt = esqueletizar(gt)
    s_pred = esqueletizar(pred)
    t_prec = float(np.logical_and(s_pred, gt).sum()) / max(float(s_pred.sum()), 1.0)
    t_sens = float(np.logical_and(s_gt, pred).sum()) / max(float(s_gt.sum()), 1.0)
    return 2.0 * t_prec * t_sens / max(t_prec + t_sens, 1e-6)


def evaluar_muestra(tinta_pred, gt):
    """Todas las métricas de una muestra (salida cruda vs. GT binario).

    La salida cruda de cada modelo puede tener la línea clara sobre negro o
    oscura sobre blanco; [a_tinta] la orienta a "tinta" antes de medir.
    """
    gt = np.asarray(gt, np.float32) > 0.5
    tinta = a_tinta(tinta_pred)
    pred_bin = binarizar(tinta)
    pred_esq = esqueletizar(pred_bin)
    gt_esq = esqueletizar(gt)
    precision, recall = _precision_recall(pred_bin, gt)
    return {
        "iou_esq": _iou(pred_esq, gt_esq),
        "dice_esq": _dice(pred_esq, gt_esq),
        "iou_bin": _iou(pred_bin, gt),
        "precision": precision,
        "recall": recall,
        "cldice": cldice(gt, pred_bin),
        "psnr": psnr(gt.astype(np.float32), tinta),
    }


METRICAS = ["cldice", "iou_esq", "dice_esq", "iou_bin", "precision", "recall", "psnr"]


def promediar(lista):
    """Promedia una lista de dictámenes de métricas (uno por muestra)."""
    if not lista:
        return {m: 0.0 for m in METRICAS}
    return {m: float(np.mean([d[m] for d in lista])) for m in METRICAS}


# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------

def cargar_muestras(npz, n, semilla=1):
    """Toma [n] pares (entrada gris uint8, GT binario) del split validación.

    El `.npz` guarda `x` (repase) y `y` (esqueleto limpio) con `split` agrupado
    por base para no tener fuga entre entrenamiento y validación.
    """
    datos = np.load(npz)
    equis, ye = datos["x"], datos["y"]
    if "split" in datos:
        split = np.asarray(datos["split"]).astype(str)
    else:
        split = np.array(["entrenamiento"] * len(equis))
    indices = np.where(split == "validacion")[0]
    if len(indices) == 0:
        indices = np.arange(len(equis))
    rng = np.random.default_rng(semilla)
    rng.shuffle(indices)
    indices = indices[:n]
    return equis[indices], ye[indices]


# --------------------------------------------------------------------------
# Modelos rivales y Buril
# --------------------------------------------------------------------------

def _clonar(url, destino):
    if not os.path.isdir(os.path.join(destino, ".git")):
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", url, destino], check=True)
    return destino


def _torch_load(ruta):
    import torch

    try:
        return torch.load(ruta, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(ruta, map_location="cpu", weights_only=False)


def _hf_descarga(repo, archivo, tipo="model"):
    from huggingface_hub import hf_hub_download

    return hf_hub_download(repo_id=repo, filename=archivo, repo_type=tipo)


def _pil_rgb(x_u8):
    from PIL import Image

    return Image.fromarray(np.stack([x_u8] * 3, axis=-1), mode="RGB")


def _dispositivo():
    import torch

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Baseline algorítmico: Zhang-Suen puro sobre el repase ------------------

def cargar_zhangsuen():
    return {"bytes": 0}


def predecir_zhangsuen(_obj, x_u8):
    tinta = a_tinta(x_u8)
    return esqueletizar(binarizar(tinta)).astype(np.float32)


# --- Buril (el modelo propio, por parches como en el móvil) -----------------

def cargar_buril(ruta):
    import tensorflow as tf

    with open(ruta, "rb") as f:
        interp = tf.lite.Interpreter(model_content=f.read())
    interp.allocate_tensors()
    return {"interp": interp, "bytes": os.path.getsize(ruta)}


def predecir_buril(obj, x_u8):
    interp = obj["interp"]
    alto, ancho = x_u8.shape
    det_in = interp.get_input_details()[0]
    det_out = interp.get_output_details()[0]
    acumulado = np.zeros((alto, ancho), np.float32)
    for ty in range(0, alto, PASO_PARQUE):
        for tx in range(0, ancho, PASO_PARQUE):
            parche = np.zeros((LADO_PARQUE, LADO_PARQUE), np.float32)
            sub = x_u8[ty:ty + LADO_PARQUE, tx:tx + LADO_PARQUE]
            parche[:sub.shape[0], :sub.shape[1]] = (sub < UMBRAL_TINTA_ENTRADA)
            interp.set_tensor(det_in["index"], parche[None, :, :, None])
            interp.invoke()
            salida = interp.get_tensor(det_out["index"])[0, :, :, 0]
            y2 = min(ty + LADO_PARQUE, alto)
            x2 = min(tx + LADO_PARQUE, ancho)
            acumulado[ty:y2, tx:x2] = np.maximum(
                acumulado[ty:y2, tx:x2], salida[:y2 - ty, :x2 - tx])
    return acumulado


# --- 1. Simo-Serra et al. (pesos .t7 convertidos a PyTorch) -----------------
# Arquitectura adaptada del código Lua original (`bobbens/sketch_simplification`)
# por el proyecto VideoComposer/ModelScope (Apache-2.0). El modelo original es
# CC-BY-NC-SA: sólo benchmark de investigación, NO comercial.

def _clase_simo_serra():
    """Devuelve la clase PyTorch del modelo (importa torch recién acá)."""
    import torch.nn as nn

    class SimoSerraSketch(nn.Module):
        """Reimplementación en PyTorch del limpiador de bocetos de Simo-Serra."""

        def __init__(self):
            super().__init__()
            self.mean = 0.9664114577640158
            self.std = 0.0858381272736797
            self.layers = nn.Sequential(
                nn.Conv2d(1, 48, 5, 2, 2), nn.ReLU(inplace=True),
                nn.Conv2d(48, 128, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(128, 128, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(128, 128, 3, 2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(256, 256, 3, 2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(256, 512, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(512, 1024, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(1024, 1024, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(1024, 1024, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(1024, 1024, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(1024, 512, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(512, 256, 3, 1, 1), nn.ReLU(inplace=True),
                nn.ConvTranspose2d(256, 256, 4, 2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(256, 128, 3, 1, 1), nn.ReLU(inplace=True),
                nn.ConvTranspose2d(128, 128, 4, 2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(128, 128, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(128, 48, 3, 1, 1), nn.ReLU(inplace=True),
                nn.ConvTranspose2d(48, 48, 4, 2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(48, 24, 3, 1, 1), nn.ReLU(inplace=True),
                nn.Conv2d(24, 1, 3, 1, 1), nn.Sigmoid())

        def forward(self, x):
            # x: [B, 1, H, W] en [0, 1], tinta oscura sobre blanco.
            return self.layers((x - self.mean) / self.std)

    return SimoSerraSketch


def _adaptar_pesos_simo_serra(sd):
    """Normaliza las claves de los pesos a la `layers.N` del modelo.

    El `.pth` convertido viene agrupado por bloques (`layer1.0`, `layer1.2`, …);
    las convoluciones, en orden, coinciden con las `layers.0/2/4/…` del
    `nn.Sequential`. Si ya vinieran planas, se devuelven tal cual.
    """
    if any(str(k).startswith("layers.") for k in sd):
        return sd
    bases = sorted({str(k).rsplit(".", 1)[0] for k in sd},
                   key=lambda b: tuple(int(p) for p in b.replace("layer", "").split(".")))
    indices = list(range(0, 45, 2))  # 23 convoluciones en el Sequential
    if len(bases) != len(indices):
        raise ValueError(f"pesos inesperados: {len(bases)} bloques, se esperaban {len(indices)}")
    nuevo = {}
    for idx, base in zip(indices, bases):
        nuevo[f"layers.{idx}.weight"] = sd[f"{base}.weight"]
        if f"{base}.bias" in sd:
            nuevo[f"layers.{idx}.bias"] = sd[f"{base}.bias"]
    return nuevo


def cargar_simo_serra():
    ruta = _hf_descarga(REPO_SIMO_SERRA, ARCHIVO_SIMO_SERRA)
    modelo = _clase_simo_serra()()
    modelo.load_state_dict(_adaptar_pesos_simo_serra(_torch_load(ruta)))
    modelo.eval()
    return {"modelo": modelo.to(_dispositivo()), "bytes": os.path.getsize(ruta)}


def predecir_simo_serra(obj, x_u8):
    import torch

    g = x_u8.astype(np.float32) / 255.0
    alto, ancho = g.shape
    ph, pw = (-alto) % 8, (-ancho) % 8
    if ph or pw:
        g = np.pad(g, ((0, ph), (0, pw)), mode="edge")
    with torch.no_grad():
        t = torch.from_numpy(g)[None, None].to(_dispositivo())
        out = obj["modelo"](t)[0, 0].cpu().numpy()
    return out[:alto, :ancho]


# --- 2. Anime2Sketch --------------------------------------------------------

def cargar_anime2sketch():
    import functools
    import importlib.util
    import torch.nn as nn

    dir_repo = _clonar(URL_ANIME2SKETCH, os.path.join(RAIZ, ".cache/benchmark/Anime2Sketch"))
    # Carga `model.py` del repo con un nombre único (evita chocar con otro `model`).
    spec = importlib.util.spec_from_file_location(
        "anime2sketch_model", os.path.join(dir_repo, "model.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    pesos = _hf_descarga("public-data/Anime2Sketch", "netG.pth")
    norm = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)
    modelo = mod.UnetGenerator(3, 1, 8, 64, norm_layer=norm, use_dropout=False)
    estado = _torch_load(pesos)
    estado = {k.replace("module.", ""): v for k, v in estado.items()}
    modelo.load_state_dict(estado)
    modelo.eval()
    return {"modelo": modelo.to(_dispositivo()), "bytes": os.path.getsize(pesos)}


def predecir_anime2sketch(obj, x_u8):
    import torch
    from torchvision.transforms import (Compose, InterpolationMode, Normalize,
                                        Resize, ToTensor)

    tf = Compose([
        Resize((LADO_DPI, LADO_DPI), InterpolationMode.BICUBIC),
        ToTensor(),
        Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    img = tf(_pil_rgb(x_u8))[None].to(_dispositivo())
    with torch.no_grad():
        out = obj["modelo"](img)[0, 0].cpu().numpy()
    return np.clip((out + 1.0) / 2.0, 0.0, 1.0)


# --- 3. Informative Drawings ------------------------------------------------
# Arquitectura `Generator` del Space `carolineec/informativedrawings`
# (MIT). Reproducida acá para cargar los pesos `model.pth` directamente.

def _clase_informative():
    """Devuelve la clase PyTorch del generador (importa torch recién acá)."""
    import torch.nn as nn

    class InformativeGenerator(nn.Module):
        """Generador ResNet de Informative Drawings (salida con sigmoide)."""

        def __init__(self, input_nc=3, output_nc=1, n_residual_blocks=3):
            super().__init__()
            norm_layer = nn.InstanceNorm2d

            class ResidualBlock(nn.Module):
                def __init__(self, canales):
                    super().__init__()
                    self.conv_block = nn.Sequential(
                        nn.ReflectionPad2d(1),
                        nn.Conv2d(canales, canales, 3),
                        norm_layer(canales),
                        nn.ReLU(inplace=True),
                        nn.ReflectionPad2d(1),
                        nn.Conv2d(canales, canales, 3),
                        norm_layer(canales),
                    )

                def forward(self, x):
                    return x + self.conv_block(x)

            self.model0 = nn.Sequential(
                nn.ReflectionPad2d(3), nn.Conv2d(input_nc, 64, 7),
                norm_layer(64), nn.ReLU(inplace=True))
            model1, entrada, salida = [], 64, 128
            for _ in range(2):
                model1 += [nn.Conv2d(entrada, salida, 3, stride=2, padding=1),
                           norm_layer(salida), nn.ReLU(inplace=True)]
                entrada, salida = salida, salida * 2
            self.model1 = nn.Sequential(*model1)
            self.model2 = nn.Sequential(
                *[ResidualBlock(entrada) for _ in range(n_residual_blocks)])
            model3 = []
            salida = entrada // 2
            for _ in range(2):
                model3 += [nn.ConvTranspose2d(entrada, salida, 3, stride=2, padding=1,
                                              output_padding=1),
                           norm_layer(salida), nn.ReLU(inplace=True)]
                entrada, salida = salida, salida // 2
            self.model3 = nn.Sequential(*model3)
            self.model4 = nn.Sequential(
                nn.ReflectionPad2d(3), nn.Conv2d(64, output_nc, 7), nn.Sigmoid())

        def forward(self, x):
            return self.model4(self.model3(self.model2(self.model1(self.model0(x)))))

    return InformativeGenerator


def cargar_informative():
    pesos = _hf_descarga(REPO_INFORMATIVE, ARCHIVO_INFORMATIVE, tipo="space")
    modelo = _clase_informative()()
    modelo.load_state_dict(_torch_load(pesos))
    modelo.eval()
    return {"modelo": modelo.to(_dispositivo()), "bytes": os.path.getsize(pesos)}


def predecir_informative(obj, x_u8):
    import torch
    from torchvision.transforms import Compose, InterpolationMode, Resize, ToTensor

    tf = Compose([Resize((LADO_DPI, LADO_DPI), InterpolationMode.BICUBIC), ToTensor()])
    img = tf(_pil_rgb(x_u8))[None].to(_dispositivo())
    with torch.no_grad():
        out = obj["modelo"](img)[0, 0].cpu().numpy()
    return out


# --- 4 y 5. ControlNet Annotators (Lineart y PidiNet) -----------------------

def cargar_controlnet(nombre):
    from controlnet_aux import LineartDetector, PidiNetDetector

    if nombre == "lineart":
        det = LineartDetector.from_pretrained("lllyasviel/Annotators")
    elif nombre == "pidinet":
        det = PidiNetDetector.from_pretrained("lllyasviel/Annotators")
    else:
        raise ValueError(nombre)
    try:
        det = det.to(_dispositivo())
    except Exception:
        pass
    return {"det": det, "modelo": nombre, "bytes": 0}


def predecir_lineart(obj, x_u8):
    salida = obj["det"](_pil_rgb(x_u8), coarse=True,
                        detect_resolution=LADO_DPI, image_resolution=LADO_DPI)
    return np.asarray(salida.convert("L"), np.float32) / 255.0


def predecir_pidinet(obj, x_u8):
    salida = obj["det"](_pil_rgb(x_u8), safe=True,
                        detect_resolution=LADO_DPI, image_resolution=LADO_DPI)
    return np.asarray(salida.convert("L"), np.float32) / 255.0


# --- Registro ---------------------------------------------------------------

def registro(buril):
    """Devuelve la lista ordenada de modelos a comparar.

    `buril` es la ruta al `.tflite`; si no existe, Buril se saltea con aviso.
    """
    modelos = []
    modelos.append({
        "id": "zhangsuen", "nombre": "Zhang-Suen", "licencia": "Algoritmo (scikit-image, BSD-3)",
        "cargar": cargar_zhangsuen, "predecir": predecir_zhangsuen,
        "exportable": True,
    })
    if buril and os.path.exists(buril):
        modelos.append({
            "id": "buril", "nombre": "Buril (nuestro)", "licencia": "Propio (datos Quick, Draw! CC BY 4.0)",
            "cargar": lambda: cargar_buril(buril), "predecir": predecir_buril,
            "exportable": True,
        })
    modelos.append({
        "id": "simoserra", "nombre": "Simo-Serra (gan)", "licencia": "CC-BY-NC-SA (NO comercial)",
        "cargar": cargar_simo_serra, "predecir": predecir_simo_serra, "exportable": False,
    })
    modelos.append({
        "id": "anime2sketch", "nombre": "Anime2Sketch", "licencia": "MIT (código)",
        "cargar": cargar_anime2sketch, "predecir": predecir_anime2sketch, "exportable": False,
    })
    modelos.append({
        "id": "informative", "nombre": "Informative Drawings", "licencia": "MIT (código)",
        "cargar": cargar_informative, "predecir": predecir_informative, "exportable": False,
    })
    modelos.append({
        "id": "lineart", "nombre": "ControlNet Lineart", "licencia": "Apache-2.0",
        "cargar": lambda: cargar_controlnet("lineart"), "predecir": predecir_lineart,
        "exportable": False,
    })
    modelos.append({
        "id": "pidinet", "nombre": "PidiNet (soft edge)", "licencia": "Apache-2.0",
        "cargar": lambda: cargar_controlnet("pidinet"), "predecir": predecir_pidinet,
        "exportable": False,
    })
    return modelos


# --------------------------------------------------------------------------
# Ejecución del benchmark (en memoria; la usan el CLI y el cuaderno)
# --------------------------------------------------------------------------

def ejecutar(npz, buril, n=8, semilla=1, ids=None, log=print):
    """Corre todos los modelos sobre [n] muestras comunes y devuelve el informe.

    Devuelve un dict con `muestras` (entradas/GT), `predicciones` por modelo y
    `resultados` (métricas + metadatos). No escribe nada en disco.
    """
    equis, yes = cargar_muestras(npz, n, semilla)
    log(f"[datos] {len(equis)} muestras de validación de {npz}")

    predicciones = {}
    informe = {"npz": npz, "muestras": len(equis), "lado": int(equis.shape[1]), "modelos": {}}

    for m in registro(buril):
        if ids and m["id"] not in ids:
            continue
        nombre = m["nombre"]
        try:
            t0 = time.time()
            obj = m["cargar"]()
            t_carga = time.time() - t0
        except Exception as e:  # noqa: BLE001
            log(f"[{nombre}] NO DISPONIBLE: {type(e).__name__}: {e}")
            continue

        por_muestra, salidas, tiempos = [], [], []
        try:
            for i in range(len(equis)):
                t0 = time.time()
                tinta = np.asarray(m["predecir"](obj, equis[i]), np.float32)
                tiempos.append((time.time() - t0) * 1000.0)
                salidas.append(tinta)
                por_muestra.append(evaluar_muestra(tinta, yes[i]))
                log(f"[{nombre}] {i + 1}/{len(equis)}  "
                    f"IoU_esq={por_muestra[-1]['iou_esq']:.3f} "
                    f"clDice={por_muestra[-1]['cldice']:.3f} "
                    f"{tiempos[-1]:.0f} ms")
        except Exception as e:  # noqa: BLE001
            log(f"[{nombre}] FALLÓ durante la inferencia: {type(e).__name__}: {e}")
            continue

        predicciones[nombre] = salidas
        informe["modelos"][nombre] = {
            "id": m["id"],
            "licencia": m["licencia"],
            "exportable_movil": m["exportable"],
            "peso_mb": obj.get("bytes", 0) / 1e6,
            "carga_s": t_carga,
            "latencia_ms": float(np.mean(tiempos)),
            "resumen": promediar(por_muestra),
            "por_muestra": por_muestra,
        }
        log(f"[{nombre}] listo: {informe['modelos'][nombre]['resumen']}")

        # Libera el modelo antes de cargar el siguiente (evita OOM con 5 pesos).
        del obj
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

    informe["orden"] = [k for k in predicciones]
    return {"entradas": equis, "gt": yes, "predicciones": predicciones, "informe": informe}


# --------------------------------------------------------------------------
# Presentación
# --------------------------------------------------------------------------

def imprimir_ranking(informe):
    """Imprime el ranking por clDice (con el resto de métricas al lado)."""
    print("\n" + "=" * 92)
    print(f"{'modelo':<24}{'clDice':>8}{'IoU_esq':>9}{'Dice_esq':>9}"
          f"{'IoU_bin':>9}{'prec':>7}{'recall':>8}{'PSNR':>7}{'ms':>7}{'MB':>8}")
    print("-" * 92)
    filas = sorted(informe["modelos"].items(),
                   key=lambda kv: kv[1]["resumen"]["cldice"], reverse=True)
    for nombre, d in filas:
        r = d["resumen"]
        print(f"{nombre:<24}{r['cldice']:>8.3f}{r['iou_esq']:>9.3f}{r['dice_esq']:>9.3f}"
              f"{r['iou_bin']:>9.3f}{r['precision']:>7.3f}{r['recall']:>8.3f}"
              f"{r['psnr']:>7.2f}{d['latencia_ms']:>7.0f}{d['peso_mb']:>8.1f}")
    print("=" * 92)


def dibujar_rejilla(entradas, gt, predicciones, ruta, max_muestras=4):
    """Rejilla PNG (entrada | cada modelo | GT) para ver dentro del cuaderno."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    nombres = list(predicciones.keys())
    n = min(max_muestras, len(entradas))
    columnas = 2 + len(nombres)
    fig, hachas = plt.subplots(n, columnas, figsize=(2.0 * columnas, 2.0 * n))
    if n == 1:
        hachas = hachas[None, :]
    for fila in range(n):
        paneles = [entradas[fila]] + [predicciones[k][fila] for k in nombres] + [gt[fila]]
        titulos = ["entrada"] + nombres + ["GT (1 px)"]
        for col in range(columnas):
            ax = hachas[fila, col]
            ax.imshow(np.asarray(paneles[col]), cmap="gray")
            ax.set_xticks([])
            ax.set_yticks([])
            if fila == 0:
                ax.set_title(titulos[col], fontsize=8)
    fig.tight_layout()
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    fig.savefig(ruta, dpi=130)
    plt.close(fig)
    return ruta


def guardar(informe, rejilla, salida):
    os.makedirs(salida, exist_ok=True)
    ruta_json = os.path.join(salida, "resultados.json")
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(informe, f, ensure_ascii=False, indent=2)
    print(f"[salida] métricas: {ruta_json}")
    if rejilla:
        print(f"[salida] rejilla:  {rejilla}")
    return ruta_json


# --------------------------------------------------------------------------
# Autoprueba (sin red ni modelos)
# --------------------------------------------------------------------------

def autoprueba():
    """Chequeos de la lógica pura (métricas, polaridad, esqueleto)."""
    from skimage.morphology import skeletonize

    linea = np.zeros((64, 64), np.float32)
    for i in range(8, 56):
        linea[i, i] = 1.0
    linea = skeletonize(linea > 0.5).astype(np.float32)
    assert 40 <= linea.sum() <= 50, f"esqueleto diagonal inesperado: {linea.sum()}"
    assert esqueletizar(linea > 0.5).sum() == linea.sum(), "esqueleto no idempotente"

    grueso = np.zeros((64, 64), np.float32)
    grueso[10:54, 10:14] = 1.0
    assert esqueletizar(grueso > 0.5).sum() < grueso.sum(), "no adelgazó la banda"

    # La misma línea, oscura sobre blanco y clara sobre negro, da la misma tinta.
    oscura = np.ones((64, 64), np.float32)
    oscura[linea > 0.5] = 0.0
    clara = np.zeros((64, 64), np.float32)
    clara[linea > 0.5] = 1.0
    assert np.allclose(a_tinta(oscura), a_tinta(clara), atol=1e-5), "polaridad"

    # Métricas: predicción perfecta = 1; ruido total = ~0.
    perfecto = evaluar_muestra(linea, linea)
    assert perfecto["iou_esq"] > 0.99 and perfecto["cldice"] > 0.99, perfecto
    vacio = evaluar_muestra(np.zeros((64, 64), np.float32), linea)
    assert vacio["cldice"] < 0.5, vacio

    print("autoprueba OK: esqueleto, polaridad y métricas funcionan.")


# --------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------



In [ ]:
# @title 5. Correr el benchmark y ver las imágenes en línea
import os

from IPython.display import Image, display

autoprueba()  # valida la lógica embebida (sin red ni modelos)

ids = [s.strip() for s in MODELOS.split(",") if s.strip()]
corrida = ejecutar(NPZ, BURIL, n=MUESTRAS, semilla=SEMILLA, ids=ids)

imprimir_ranking(corrida["informe"])

if corrida["predicciones"]:
    salida = os.path.join(os.getcwd(), ".cache/benchmark")
    guardar(corrida["informe"], None, salida)
    try:
        ruta_rejilla = dibujar_rejilla(
            corrida["entradas"], corrida["gt"], corrida["predicciones"],
            os.path.join(salida, "rejilla.png"),
            max_muestras=min(4, MUESTRAS))
        display(Image(ruta_rejilla))
    except Exception as e:
        print("No se pudo dibujar la rejilla:", type(e).__name__, e)
else:
    print("No se pudo evaluar ningún modelo.")


In [ ]:
# @title 6. (Opcional) Interfaz Gradio para comparar muestra por muestra
import numpy as np

if "corrida" in globals() and corrida["predicciones"]:
    nombres = ["entrada"] + corrida["informe"]["orden"] + ["GT (1 px)"]

    def _ver(a):
        a = np.asarray(a, np.float32)
        if a.max() > 1.0:
            a = a / 255.0
        return (a * 255).astype(np.uint8)

    def _galeria(i):
        i = int(i)
        imgs = ([corrida["entradas"][i]]
                + [corrida["predicciones"][k][i] for k in corrida["informe"]["orden"]]
                + [corrida["gt"][i]])
        return [(_ver(a), t) for a, t in zip(imgs, nombres)]

    try:
        import gradio as gr
    except ImportError:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
        import gradio as gr

    with gr.Blocks() as demo:
        gr.Markdown("## Limpieza de trazos — entrada → modelos → ground truth")
        deslizador = gr.Slider(0, max(len(corrida["entradas"]) - 1, 0),
                               step=1, value=0, label="muestra")
        galeria = gr.Gallery(label="comparación")
        demo.load(_galeria, deslizador, galeria)
        deslizador.change(_galeria, deslizador, galeria)
    demo.launch(debug=False, share=True)
else:
    print("Corré primero la celda 5.")


## Cómo leer los resultados

- **clDice** es la métrica más informativa acá: con *ground truth* de 1 px el IoU
  es durísimo y castiga a todos por igual; clDice premia seguir la **conectividad**
  correcta.
- **IoU/Dice esqueleto↔esqueleto** mide la forma fina; **IoU de máscara** favorece
  a las líneas gruesas.
- **Latencia** y **peso** son la ventaja real de Buril (sin red, en el equipo).
- La rejilla y la interfaz Gradio son para inspección visual: una métrica no
  reemplaza el ojo.

## Próximos pasos

Con las métricas reales se puede cerrar el pendiente de `ROADMAP.md`
("ajustar épocas/semilla") y decidir si conviene ajustar el `--parches-por-imagen`
o la aumentación. Si un rival no carga (por ejemplo, por pesos movidos), el
cuaderno lo reporta y sigue con los demás.